# Counterfactual, Transportable Risk Prediction -- Simulation Study

This notebook implements simulation experiments for the method in `note.md`.

**Goal.** Estimate $\\beta$ in
$$E(Y^0 \\mid P,\\, S=0) = \\beta_0 + \\beta_1 P_1 + \\beta_2 P_2$$
where $P = (X_1, X_2)$, by combining an external cohort with a target population.

**Three estimators:**
1. **Plug-in (3-step):** outcome regression -> marginalise over $L$ -> regress on $P$
2. **DR estimator:** doubly-robust estimating equation $\\mathbb{P}_n \\phi_\\beta = 0$
3. **Oracle DR:** DR with true nuisance values (gold-standard benchmark)

**The DR estimator has a two-condition double robustness (Corollary):**

> $\\hat\\beta$ is consistent when **both** of the following hold simultaneously:
> - **Condition 1:** either $m$ or $\\pi$ is correctly specified
> - **Condition 2:** either $\\psi$ or $\\rho_0$ is correctly specified

**Section 5** tests Condition 1 (varying $m$ and $\\pi$, with $\\psi$/$\\rho$ always correct).
**Section 8** tests Condition 2 (varying $\\psi$ and $\\rho$, with $m$/$\\pi$ always correct).


## 1. Imports

In [18]:
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.linear_model import LinearRegression, LogisticRegression
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from joblib import Parallel, delayed

print('Imports OK')

Imports OK


## 2. Data-Generating Process (DGP)

### Structural Equations

**Covariates** (both populations):
$$X_1 \sim N(0,1),\quad X_2 \sim N(0,1),\quad X_1 \perp X_2$$

Risk predictors $P = (X_1, X_2)^\top$; same as transportability covariates $X^{(1)} = (X_1, X_2)^\top$.

**Cohort indicator** (depends on both $X_1$ and $X_2$):
$$S \mid X_1, X_2 \sim \text{Bernoulli}\!\left(\text{expit}(-0.5 + 0.4\,X_1 + 0.4\,X_2)\right)$$

**Confounder** (generated for all; observed only in cohort):
$$L = 0.5\,X_1 + 0.5\,X_2 + \varepsilon_L,\quad \varepsilon_L \sim N(0,1)$$

**Treatment** (cohort only, $S=1$):
$$A \mid X, L, S=1 \sim \text{Bernoulli}\!\left(\text{expit}(-1 + 0.5\,X_1 + 1.0\,L)\right)$$

**Outcome** (cohort only, $S=1$):
$$Y = 1 + 0.5\,X_1 + 0.3\,X_2 + 0.8\,L + 1.5\,A + \varepsilon_Y,\quad \varepsilon_Y \sim N(0,1)$$

### True parameter

Marginalising $L$ out of the potential outcome $Y^0 = 1 + 0.5X_1 + 0.3X_2 + 0.8L + \varepsilon_Y$:

$$E(Y^0 \mid X_1, X_2) = 1 + 0.5X_1 + 0.3X_2 + 0.8 \underbrace{E(L \mid X_1,X_2)}_{=\,0.5X_1+0.5X_2}
= 1 + 0.9\,X_1 + 0.7\,X_2$$

Because $\varepsilon_L, \varepsilon_Y \perp S \mid (X_1,X_2)$ and $L \perp S \mid (X_1,X_2)$, conditioning on $S=0$ does not change this:

$$\boxed{E(Y^0 \mid X_1, X_2, S=0) = 1 + 0.9\,X_1 + 0.7\,X_2}$$

so $\beta^* = (\beta_0^*, \beta_1^*, \beta_2^*) = (1,\; 0.9,\; 0.7)$.

### Assumption verification

| Assumption | How satisfied |
|---|---|
| **A1** Consistency | $Y = Y^A$ by construction |
| **A2** No unmeasured confounding | $Y^0 \perp A \mid L, X, S=1$: given $(X,L)$, $A$ is Bernoulli noise independent of $\varepsilon_Y$ |
| **A3** Positivity of treatment | $\pi^*(X,L) = \text{expit}(\cdot) \in (0,1)$ for all finite $(X,L)$ |
| **A4** Transportability | $Y^0 \perp S \mid (X_1,X_2)$: $S$ is Bernoulli noise given $(X_1,X_2)$, independent of $\varepsilon_L,\varepsilon_Y$ |
| **A5** Positivity of membership | $\rho_1^*(X) = \text{expit}(-0.5+0.4X_1+0.4X_2) \in (0,1)$ |


In [19]:
# ── True parameter  β* = (β0, β1, β2) ────────────────────────────
BETA_TRUE = np.array([1.0, 0.9, 0.7])   # intercept, slope X1, slope X2

# ── True nuisance functions ───────────────────────────────────────
def m_true(X1, X2, L):
    """E(Y | A=0, X, L, S=1)"""
    return 1 + 0.5*X1 + 0.3*X2 + 0.8*L

def pi_true(X1, L):
    """Pr(A=1 | X, L, S=1)"""
    return expit(-1 + 0.5*X1 + 1.0*L)

def psi_true(X1, X2):
    """E{m(X,L) | X, S=1} = 1 + 0.9*X1 + 0.7*X2"""
    return 1 + 0.9*X1 + 0.7*X2

def rho1_true(X1, X2):
    """Pr(S=1 | X1, X2)"""
    return expit(-0.5 + 0.4*X1 + 0.4*X2)

def rho0_true(X1, X2):
    """Pr(S=0 | X1, X2)"""
    return 1 - rho1_true(X1, X2)


# ── DGP ──────────────────────────────────────────────────────────
def generate_data(n, seed=None):
    """
    Generate one dataset of size n.

    Columns:
      X1, X2   – covariates (both populations)
      P1, P2   – risk predictors = X1, X2 (both populations)
      S        – cohort indicator: 0 = target, 1 = cohort
      L        – confounder (observed only when S=1; set to NaN for S=0)
      A        – treatment  (observed only when S=1; NaN for S=0)
      Y        – outcome    (observed only when S=1; NaN for S=0)
      Y0_true  – true Y^0  (evaluation only, never used in estimation)
    """
    rng = np.random.default_rng(seed)

    # covariates
    X1 = rng.standard_normal(n)
    X2 = rng.standard_normal(n)

    # cohort indicator – depends on BOTH X1 and X2  (A4 still holds
    # because S ⊥ ε_L, ε_Y | X1, X2)
    S = rng.binomial(1, expit(-0.5 + 0.4*X1 + 0.4*X2))

    # confounder  (L ⊥ S | X, so transportability is preserved)
    eps_L = rng.standard_normal(n)
    L = 0.5*X1 + 0.5*X2 + eps_L

    # treatment
    prob_A = expit(-1 + 0.5*X1 + 1.0*L)
    A_full = rng.binomial(1, prob_A)

    # outcome
    eps_Y = rng.standard_normal(n)
    Y_full = 1 + 0.5*X1 + 0.3*X2 + 0.8*L + 1.5*A_full + eps_Y

    # true counterfactual Y^0 (for evaluation only)
    Y0_true = 1 + 0.5*X1 + 0.3*X2 + 0.8*L + eps_Y

    # mask cohort-only variables for the target population
    A = np.where(S == 1, A_full.astype(float), np.nan)
    Y = np.where(S == 1, Y_full,               np.nan)
    L_obs = np.where(S == 1, L,                np.nan)

    return pd.DataFrame({
        'X1': X1, 'X2': X2,
        'P1': X1, 'P2': X2,     # risk predictors
        'S':  S,
        'L':  L_obs,
        'L_full': L,             # kept for oracle nuisance only
        'A':  A,
        'Y':  Y,
        'Y0_true': Y0_true,
    })


# ── Quick look ────────────────────────────────────────────────────
df = generate_data(n=5000, seed=0)
n0 = (df.S == 0).sum(); n1 = (df.S == 1).sum()
print(f'Sample sizes  →  target (S=0): {n0},  cohort (S=1): {n1}')
print()
print(df.groupby('S')[['X1','X2','L','A','Y']].mean().round(3).rename(
    index={0:'S=0 mean', 1:'S=1 mean'}))

Sample sizes  →  target (S=0): 3067,  cohort (S=1): 1933

             X1     X2      L      A      Y
S                                          
S=0 mean -0.139 -0.125    NaN    NaN    NaN
S=1 mean  0.209  0.242  0.239  0.383  1.943


### 2.2  Verify the true $\beta^*$ numerically

Confirm $E(Y^0 \mid P, S=0) = 1 + 0.9X_1 + 0.7X_2$ by OLS of $Y^{0}$ on $(1, X_1, X_2)$ in the large target population.

## 3. Estimators

### 3.1  Plug-in (3-step) estimator

1. **Step 1** — Fit $\hat m(X,L)$ for $E(Y \mid A=0, X, L, S=1)$ in cohort via linear regression.
2. **Step 2** — Predict $\hat m_j$ for all cohort members; regress on $(X_1,X_2)$ to get $\hat\psi(X)$.
3. **Step 3** — Predict $\hat\psi_i$ for target population; regress on $(1, X_1, X_2)$ to get $\hat\beta$.

In [20]:
def estimator_plugin(df, misspecify_m=False, misspecify_psi=False):
    """
    3-step plug-in estimator.
    misspecify_m=True:   omit L from the outcome model (Step 1).
    misspecify_psi=True: use X2 only (omit X1) when marginalising over L (Step 2).
    Returns beta_hat = [beta0, beta1, beta2].
    """
    cohort  = df[df.S == 1].copy()
    target  = df[df.S == 0].copy()
    cohort0 = cohort[cohort.A == 0]

    # Step 1: outcome model on A=0 cohort rows
    feat_m = ['X1'] if misspecify_m else ['X1', 'X2', 'L']
    lm_m = LinearRegression().fit(cohort0[feat_m], cohort0.Y)

    # Step 2: marginalise over L by regressing m_hat on X in cohort
    #         misspecify_psi=True -> use X2 only, omitting X1
    m_hat_cohort = lm_m.predict(cohort[feat_m])
    feat_psi = ['X2'] if misspecify_psi else ['X1', 'X2']
    lm_psi = LinearRegression().fit(cohort[feat_psi], m_hat_cohort)

    # Step 3: predict psi in target, regress on P = (X1, X2)
    psi_hat_target = lm_psi.predict(target[feat_psi])
    lm_beta = LinearRegression().fit(target[['P1', 'P2']], psi_hat_target)

    return np.array([lm_beta.intercept_, *lm_beta.coef_])


### 3.2  Doubly-Robust (DR) estimator

Solves $\mathbb{P}_n\,\phi_\beta = 0$ with instrument $d(P) = (1, P_1, P_2)^\top$:

$$\frac{1}{n}\sum_i d(P_i)\!\left[
  (1-S_i)\bigl\{\hat\psi(X_i) - \hat\beta^\top d(P_i)\bigr\}
  + S_i\frac{\hat\rho_0(X_i)}{\hat\rho_1(X_i)}
  \left\{\frac{1-A_i}{1-\hat\pi_i}(Y_i-\hat m_i) + \hat m_i - \hat\psi(X_i)\right\}
\right] = 0$$

This is **linear in $\beta$** and has the closed-form solution:
$$\hat\beta
= \underbrace{\left[\sum_{i:\,S_i=0} d(P_i)d(P_i)^\top\right]^{-1}}_{V^{-1}}
  \sum_i d(P_i)\!\left[
    (1-S_i)\hat\psi(X_i)
    + S_i\frac{\hat\rho_0}{\hat\rho_1}\left\{\frac{1-A_i}{1-\hat\pi_i}(Y_i-\hat m_i)+\hat m_i-\hat\psi(X_i)\right\}
  \right]$$

In [21]:
def fit_nuisance(df,
                 misspecify_m=False,
                 misspecify_pi=False,
                 misspecify_psi=False,
                 misspecify_rho=False):
    """
    Estimate all four nuisance functions from data.

    misspecify_m   - omit L from the outcome model
    misspecify_pi  - omit L from the propensity model
    misspecify_psi - omit X1 from the psi (marginal outcome) model [uses X2 only]
    misspecify_rho - omit X2 from the cohort-membership model

    Uses L_full (NaN-free) so predictions work for the full dataset.
    Returns a dict of arrays indexed to the full df.
    """
    cohort  = df[df.S == 1].copy()
    cohort0 = cohort[cohort.A == 0]

    # m: E(Y | A=0, X, L, S=1)
    feat_m = ['X1', 'X2'] if misspecify_m else ['X1', 'X2', 'L_full']
    lm_m   = LinearRegression().fit(cohort0[feat_m], cohort0.Y)
    m_hat  = lm_m.predict(df[feat_m])

    # pi: Pr(A=1 | X, L, S=1)
    feat_pi = ['X1'] if misspecify_pi else ['X1', 'L_full']
    lr_pi   = LogisticRegression(max_iter=500).fit(
                  cohort[feat_pi], cohort.A.astype(int))
    pi_hat  = np.clip(lr_pi.predict_proba(df[feat_pi])[:, 1], 1e-6, 1-1e-6)

    # rho1: Pr(S=1 | X)
    # misspecify_rho=True -> use X1 only, omitting X2
    feat_rho = ['X1'] if misspecify_rho else ['X1', 'X2']
    lr_rho   = LogisticRegression(max_iter=500).fit(df[feat_rho], df.S)
    rho1_hat = np.clip(lr_rho.predict_proba(df[feat_rho])[:, 1], 1e-6, 1-1e-6)
    rho0_hat = 1 - rho1_hat

    # psi: E{m(X,L) | X, S=1}
    # misspecify_psi=True -> use X2 only, omitting X1
    m_hat_cohort = lm_m.predict(cohort[feat_m])
    feat_psi = ['X2'] if misspecify_psi else ['X1', 'X2']
    lm_psi   = LinearRegression().fit(cohort[feat_psi], m_hat_cohort)
    psi_hat  = lm_psi.predict(df[feat_psi])

    return {
        'm_hat':    m_hat,
        'pi_hat':   pi_hat,
        'rho0_hat': rho0_hat,
        'rho1_hat': rho1_hat,
        'psi_hat':  psi_hat,
    }


def estimator_dr(df, nuisance=None, oracle=False, **kwargs):
    """
    DR estimator (closed-form).
    oracle=True plugs in the true nuisance functions.
    Returns beta_hat = [beta0, beta1, beta2].
    """
    S  = df.S.values
    X1 = df.X1.values; X2 = df.X2.values
    A  = df.A.fillna(0).values
    Y  = df.Y.fillna(0).values
    L  = df.L_full.values
    n  = len(df)

    if oracle:
        m    = m_true(X1, X2, L)
        pi   = pi_true(X1, L)
        psi  = psi_true(X1, X2)
        rho0 = rho0_true(X1, X2)
        rho1 = rho1_true(X1, X2)
    else:
        if nuisance is None:
            nuisance = fit_nuisance(df, **kwargs)
        m    = nuisance['m_hat']
        pi   = nuisance['pi_hat']
        psi  = nuisance['psi_hat']
        rho0 = nuisance['rho0_hat']
        rho1 = nuisance['rho1_hat']

    D = np.column_stack([np.ones(n), X1, X2])
    aipw    = ((1 - A) / (1 - pi)) * (Y - m) + m - psi
    contrib = (1 - S) * psi  +  S * (rho0 / rho1) * aipw

    numerator   = D.T @ contrib
    denominator = (D * (1 - S)[:, None]).T @ D
    return np.linalg.solve(denominator, numerator)


## 4. Single-dataset sanity check

## 5. Monte Carlo simulation

Repeat over `N_SIM` replications and record bias, RMSE, and 95 % CI coverage.

In [22]:
def _one_rep(i, n_obs, base_seed, oracle, scenario):
    """Single Monte Carlo replication — runs in a parallel thread."""
    kw_keys = ('misspecify_m', 'misspecify_pi', 'misspecify_psi', 'misspecify_rho')
    kw      = {k: scenario[k] for k in kw_keys if k in scenario}

    df_i   = generate_data(n=n_obs, seed=base_seed + i)
    b_plug = estimator_plugin(
                 df_i,
                 misspecify_m=scenario.get('misspecify_m', False),
                 misspecify_psi=scenario.get('misspecify_psi', False))

    nus  = None if oracle else fit_nuisance(df_i, **kw)
    b_dr = estimator_dr(df_i, nuisance=nus, oracle=oracle)

    return {
        'plug_b0': b_plug[0], 'plug_b1': b_plug[1], 'plug_b2': b_plug[2],
        'dr_b0':   b_dr[0],   'dr_b1':   b_dr[1],   'dr_b2':   b_dr[2],
    }


def run_simulation(n_obs, n_sim, scenario, base_seed=0, n_jobs=-1):
    """
    Run n_sim Monte Carlo replications in parallel (n_jobs=-1 = all CPUs).
    Uses threading backend so numpy/sklearn GIL-releases give real speedup.

    scenario keys: misspecify_m, misspecify_pi, misspecify_psi,
                   misspecify_rho, use_oracle  (all bool, default False).
    Returns a DataFrame with per-replication estimates.
    """
    oracle  = scenario.get('use_oracle', False)
    records = Parallel(n_jobs=n_jobs, backend='threading')(
        delayed(_one_rep)(i, n_obs, base_seed, oracle, scenario)
        for i in range(n_sim)
    )
    return pd.DataFrame(records)


def summarise(res, label):
    out = {'label': label}
    for coef, truth in [('b1', BETA_TRUE[1]), ('b2', BETA_TRUE[2])]:
        out[f'pl_{coef}_bias'] = (res[f'plug_{coef}'] - truth).mean()
        out[f'pl_{coef}_rmse'] = np.sqrt(((res[f'plug_{coef}'] - truth) ** 2).mean())
        out[f'dr_{coef}_bias'] = (res[f'dr_{coef}'] - truth).mean()
        out[f'dr_{coef}_rmse'] = np.sqrt(((res[f'dr_{coef}'] - truth) ** 2).mean())
    return out


In [23]:
import itertools

N_OBS = 2000
N_SIM = 500

flags = ['m', 'pi', 'psi', 'rho']
keys  = ['misspecify_m', 'misspecify_pi', 'misspecify_psi', 'misspecify_rho']

def make_label(vals):
    sym = {True: '\u2717', False: '\u2713'}   # ✗ / ✓
    return '  '.join(f'{f}{sym[v]}' for f, v in zip(flags, vals))

all_scenarios = [
    (make_label(combo), dict(zip(keys, combo)))
    for combo in itertools.product([False, True], repeat=4)
]

all_summaries = []

print(f"Running {len(all_scenarios)} scenarios x {N_SIM} reps each "
      f"(n={N_OBS}) in parallel ...")
for label, scen in all_scenarios:
    res = run_simulation(N_OBS, N_SIM, scen)
    all_summaries.append(summarise(res, label))

print("Done.")


Running: All correct ... 
-- All correct --
                               Bias(b1)   RMSE(b1)    Cov%  |   Bias(b2)   RMSE(b2)    Cov%
  ------------------------------------------------------------------------------
  Plug-in                       +0.0049     0.0589    0.0%  |    +0.0050     0.0563    0.0%
  DR                            +0.0018     0.0689   93.4%  |    +0.0020     0.0704   93.6%
Running: Misspecify m ... 
-- Misspecify m --
                               Bias(b1)   RMSE(b1)    Cov%  |   Bias(b2)   RMSE(b2)    Cov%
  ------------------------------------------------------------------------------
  Plug-in                       -0.1657     0.1800    0.0%  |    -0.7000     0.7000    0.0%
  DR                            -0.0014     0.0814   94.6%  |    +0.0013     0.0810   95.2%
Running: Misspecify π ... 
-- Misspecify π --
                               Bias(b1)   RMSE(b1)    Cov%  |   Bias(b2)   RMSE(b2)    Cov%
  --------------------------------------------------------

## 6. Results: double-robustness table (all 16 combinations)

In [24]:
df16 = pd.DataFrame(all_summaries).set_index('label')

disp = df16[[
    'pl_b1_bias', 'pl_b1_rmse',
    'dr_b1_bias', 'dr_b1_rmse',
    'pl_b2_bias', 'pl_b2_rmse',
    'dr_b2_bias', 'dr_b2_rmse',
]].copy()

disp.columns = [
    'Plug Bias(\u03b21)', 'Plug RMSE(\u03b21)',
    'DR Bias(\u03b21)',   'DR RMSE(\u03b21)',
    'Plug Bias(\u03b22)', 'Plug RMSE(\u03b22)',
    'DR Bias(\u03b22)',   'DR RMSE(\u03b22)',
]

# Format: bias with sign, RMSE plain, 4 decimals
def fmt_bias(x): return f'{x:+.4f}'
def fmt_rmse(x): return f'{x:.4f}'

styled = disp.copy().astype(str)
for c in disp.columns:
    if 'Bias' in c:
        styled[c] = disp[c].map(fmt_bias)
    else:
        styled[c] = disp[c].map(fmt_rmse)

print(styled.to_string())


                   Plug Bias(β1)  Plug RMSE(β1)  DR Bias(β1)  DR RMSE(β1) DR Cov%(β1)  Plug Bias(β2)  Plug RMSE(β2)  DR Bias(β2)  DR RMSE(β2) DR Cov%(β2)
label                                                                                                                                                    
All correct               0.0049         0.0589       0.0018       0.0689       93.4%          0.005         0.0563       0.0020       0.0704       93.6%
Misspecify m             -0.1657         0.1800      -0.0014       0.0814       94.6%         -0.700         0.7000       0.0013       0.0810       95.2%
Misspecify π              0.0049         0.0589       0.0022       0.0671       94.4%          0.005         0.0563       0.0022       0.0701       93.8%
Both misspecified        -0.1657         0.1800      -0.0987       0.1210       69.8%         -0.700         0.7000      -0.0471       0.0855       90.0%
Oracle                    0.0049         0.0589       0.0013       0.0684   